# Setup dell'Ambiente e Gestione del Dataset
In questa cella colleghiamo Google Drive, ci spostiamo nella cartella di lavoro e prepariamo il dataset scompattandolo nello spazio locale di Colab per velocizzare la lettura delle immagini.

In [ ]:
from google.colab import drive
import os

# 1. Mount del Drive e posizionamento nella cartella del progetto
drive.mount('/content/drive')
os.chdir('/content/drive/Shareddrives/ML_Project/Data/')

# 2. Copia e scompattamento del dataset nello spazio locale temporaneo di Colab
!cp "./waste_type_identification.zip" "/content/"
!unzip -q "/content/waste_type_identification.zip" -d "/content/dataset/"

print("Dataset caricato e scompattato localmente con successo.")

Mounted at /content/drive
Dataset caricato e scompattato localmente con successo.


# Collegamento dei File Esterni
Utilizziamo il comando %run per caricare in memoria tutto ciò che hai scritto negli altri due notebook. In questo modo, le funzioni get_dataloaders, initial_reset, train_model_decay_lr e plot_metrics saranno immediatamente utilizzabili.

In [ ]:
# Importazione magica delle funzioni e delle classi dagli altri notebook
%run "/content/drive/Shareddrives/ML_Project/ResNet/data_augmentation.ipynb"
%run "/content/drive/Shareddrives/ML_Project/ResNet/training_hyperparameters.ipynb"

print("Notebook esterni agganciati e pronti all'uso.")

Notebook esterni agganciati e pronti all'uso.


# Definizione degli Iperparametri Globali
Qui centralizziamo tutte le variabili modificabili. Se vuoi cambiare la configurazione dei tuoi test (griglia di iperparametri), ti basterà modificare esclusivamente i valori in questa cella.

In [ ]:
# Definizione delle variabili di configurazione e degli iperparametri
DATA_DIR = '/content/dataset/waste_type_identification'
BATCH_SIZE = 32
NUM_CLASSES = 8
NUM_EPOCHS = 15          # Numero di epoche totali per il test
LEARNING_RATE = 0.001     # Learning rate di partenza
TRAINING_LAYERS = 1       # 0 = Solo ultimo layer (FC), 1 = Layer 4 + FC, ecc.

# Inizializzazione dei Flussi Dati (Dataloaders)
Invochiamo la funzione contenuta in data_augmentation.ipynb per generare i dizionari di addestramento e validazione strutturati con il campionamento pesato e il wrapper condizionale.

In [ ]:
# Generazione dei Dataloaders e recupero dei nomi delle classi
dataloaders, class_names = get_dataloaders(
    data_dir=DATA_DIR,
    batch_size=BATCH_SIZE,
    val_size=0.2,
    random_seed=42
)

Campioni medi per classe: 1552
Classi minoritarie identificate (sottoposte ad Augmentation):
- battery (756 img in train)
- metal (615 img in train)
- organic (788 img in train)
- plastic (692 img in train)
- undifferentiated (558 img in train)

Dataloader pronti.


# Calcolo dei Pesi per la Loss Funzionale
Dato che il metodo train_model_decay_lr accetta un parametro opzionale class_weights, estraiamo la distribuzione reale dei target di addestramento per bilanciare il calcolo della Cross Entropy.

In [ ]:
import numpy as np
import torch

# Estrazione delle etichette reali del subset di training per calcolare i pesi della loss
train_subset = dataloaders['train'].dataset.subset
train_targets = [train_subset.dataset.targets[i] for i in train_subset.indices]

# Calcolo delle frequenze e inversione dei pesi
class_counts = np.bincount(train_targets, minlength=NUM_CLASSES)
class_weights = 1.0 / class_counts.astype(float)
class_weights_normalization = class_weights / class_weights.sum()

# Conversione in tensore PyTorch
loss_weights_tensor = torch.FloatTensor(class_weights_normalization)
print(f"Pesi calcolati per il bilanciamento della Loss: {loss_weights_tensor}")

Pesi calcolati per il bilanciamento della Loss: tensor([0.1488, 0.0193, 0.0699, 0.1829, 0.1427, 0.0724, 0.1625, 0.2016])


# Inizializzazione della Rete, Ottimizzatore e Scheduler
Prepariamo l'architettura della ResNet-18 e configuriamo il meccanismo di decadimento del Learning Rate. Poiché la funzione nel tuo notebook esegue scheduler.step() senza passare parametri a fine epoca, utilizzeremo uno scheduler basato sul conteggio delle epoche (come StepLR).

In [ ]:
import torch.optim as optim

# 1. Inizializzazione del modello tramite la funzione di training_hyperparameters
model = initial_reset(num_classes=NUM_CLASSES, training_layers=TRAINING_LAYERS)

# 2. Configurazione dell'ottimizzatore Adam (passando solo i parametri sbloccati che richiedono gradiente)
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LEARNING_RATE)

# 3. Configurazione dello Scheduler: riduce il LR moltiplicandolo per 'gamma' (0.1) ogni 'step_size' (5) epoche
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)

print(f"Modello pronto. Layer ottimizzabili configurati con successo.")

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 142MB/s]


Modello pronto. Layer ottimizzabili configurati con successo.


# Esecuzione dell'Addestramento e Visualizzazione delle Metriche
Avviamo il motore di calcolo del secondo notebook e passiamo l'output direttamente alla funzione grafica per valutare visivamente l'andamento delle performance

In [ ]:
# Avvio del ciclo di addestramento principale
model_addestrato, history = train_model_decay_lr(
    model=model,
    dataloaders=dataloaders,
    optimizer=optimizer,
    epochs=NUM_EPOCHS,
    scheduler=scheduler,
    class_weights=loss_weights_tensor
)

# Renderizzazione dei grafici di Loss, Accuracy e Balanced Accuracy
plot_metrics(history)


Epoca 1/15


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


# Salvataggio Finale del Modello Ottimizzato
Fase finale di persistenza per memorizzare l'architettura e i pesi migliori ottenuti su Google Drive.

In [ ]:
# Salvataggio del modello addestrato nella cartella dedicata alle sessioni di training
percorso_salvataggio = '/content/drive/Shareddrives/ML_Project/TrainingSession/resnet18_main_trained.pth'
torch.save(model_addestrato, percorso_salvataggio)

print(f"Sessione completata con successo! Modello esportato in: {percorso_salvataggio}")